In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 필수 라이브러리 업데이트 (for miniconda)
!apt-get update
!apt-get install -y build-essential cmake ninja-build git wget
!pip install -q plyfile

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda

import os
os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]
!conda --version

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,435 kB]
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,852 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-dr

In [ ]:
# WeatherGS clone
!git clone --recursive https://github.com/Jumponthemoon/WeatherGS.git
%cd WeatherGS

# 서브모듈 직접 클론
%cd 3DGS

!git clone https://github.com/graphdeco-inria/diff-gaussian-rasterization submodules/diff-gaussian-rasterization
!git clone https://gitlab.inria.fr/bkerbl/simple-knn.git submodules/simple-knn


Cloning into 'WeatherGS'...
remote: Enumerating objects: 819, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 819 (delta 32), reused 3 (delta 3), pack-reused 773 (from 3)
Receiving objects: 100% (819/819), 175.24 MiB | 40.31 MiB/s, done.
Resolving deltas: 100% (457/457), done.
Updating files: 100% (69/69), done.
/content/WeatherGS
/content/WeatherGS/3DGS
Cloning into 'submodules/diff-gaussian-rasterization'...
remote: Enumerating objects: 329, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 329 (delta 183), reused 177 (delta 177), pack-reused 124 (from 1)
Receiving objects: 100% (329/329), 112.49 KiB | 1.81 MiB/s, done.
Resolving deltas: 100% (219/219), done.
Cloning into 'submodules/simple-knn'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 37 (delt

In [ ]:
# g++-10 설치 (CUDA 11.6을 위해)
!apt-get install -y g++-10

# 버전 확인
!g++-10 --version

# 환경설정 세팅 (submodule은 설치 안될것임)
%cd /content/WeatherGS/3DGS
!conda env create -f environment.yml -n weathergs

# CUDA 11.6 설치
!conda run --no-capture-output -n weathergs conda install -c conda-forge cudatoolkit-dev=11.6 -y

# 버전 확인, release 11.6 이 나와야 함
!conda run -n weathergs nvcc --version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  cpp-10 gcc-10 gcc-10-base libgcc-10-dev libstdc++-10-dev
Suggested packages:
  gcc-10-locales g++-10-multilib gcc-10-doc gcc-10-multilib libstdc++-10-doc
The following NEW packages will be installed:
  cpp-10 g++-10 gcc-10 gcc-10-base libgcc-10-dev libstdc++-10-dev
0 upgraded, 6 newly installed, 0 to remove and 122 not upgraded.
Need to get 43.7 MB of archives.
After this operation, 142 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 gcc-10-base amd64 10.5.0-1ubuntu1~22.04.3 [213 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 cpp-10 amd64 10.5.0-1ubuntu1~22.04.3 [9,396 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libgcc-10-dev amd64 10.5.0-1ubuntu1~22.04.3 [2,493 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/unive

In [ ]:
!cd /content/WeatherGS/3DGS/submodules/diff-gaussian-rasterization && \
    git submodule update --init --recursive

# g++-10을 사용해서 Submodule build
!conda run --no-capture-output -n weathergs bash -c "\
    cd /content/WeatherGS/3DGS/submodules/diff-gaussian-rasterization && \
    export CXX=g++-10 && \
    export CC=gcc-10 && \
    python setup.py install 2>&1 | tail -20"

!conda run --no-capture-output -n weathergs bash -c "\
    cd /content/WeatherGS/3DGS/submodules/simple-knn && \
    export CXX=g++-10 && \
    export CC=gcc-10 && \
    python setup.py install 2>&1 | tail -20"

In [ ]:
# factory_rain

!conda run --no-capture-output -n weathergs python train.py \
    -s /content/drive/MyDrive/WeatherGS_test/final_scenes/factory_rain/ \
    --masks /content/drive/MyDrive/WeatherGS_test/final_scenes/factory_rain/masks \
    -m ./output/factory_rain_cleaned

!cp -r /content/WeatherGS/3DGS/output/factory_rain_cleaned /content/drive/MyDrive/WeatherGS_test/output/factory_rain_cleaned/


Optimizing ./output/factory_rain_cleaned
Output folder: ./output/factory_rain_cleaned [13/03 04:58:38]
Tensorboard not available: not logging progress [13/03 04:58:38]
Reading camera 1/241 22222 [13/03 04:58:41]
Camera(id=1, model='PINHOLE', width=1200, height=800, params=array([1534.68704259, 1531.51038844,  600.        ,  400.        ])) [13/03 04:58:41]
PINHOLE [13/03 04:58:41]
Reading camera 2/241 22222 [13/03 04:58:42]
Camera(id=1, model='PINHOLE', width=1200, height=800, params=array([1534.68704259, 1531.51038844,  600.        ,  400.        ])) [13/03 04:58:42]
PINHOLE [13/03 04:58:42]
Reading camera 3/241 22222 [13/03 04:58:42]
Camera(id=1, model='PINHOLE', width=1200, height=800, params=array([1534.68704259, 1531.51038844,  600.        ,  400.        ])) [13/03 04:58:42]
PINHOLE [13/03 04:58:42]
Reading camera 4/241 22222 [13/03 04:58:43]
Camera(id=1, model='PINHOLE', width=1200, height=800, params=array([1534.68704259, 1531.51038844,  600.        ,  400.        ])) [13/03 04:

In [ ]:
# rendering

!conda run --no-capture-output -n weathergs python render.py -m /content/drive/MyDrive/WeatherGS_test/output/tanabata_snow_cleaned --eval
!conda run --no-capture-output -n weathergs python render.py -m /content/drive/MyDrive/WeatherGS_test/output/tanabata_rain_cleaned --eval
!conda run --no-capture-output -n weathergs python render.py -m /content/drive/MyDrive/WeatherGS_test/output/pool_snow_cleaned --eval
!conda run --no-capture-output -n weathergs python render.py -m /content/drive/MyDrive/WeatherGS_test/output/pool_rain_cleaned --eval
!conda run --no-capture-output -n weathergs python render.py -m /content/drive/MyDrive/WeatherGS_test/output/factory_snow_cleaned --eval
!conda run --no-capture-output -n weathergs python render.py -m /content/drive/MyDrive/WeatherGS_test/output/factory_rain_cleaned --eval


Looking for config file in /content/drive/MyDrive/WeatherGS_test/output/tanabata_rain_cleaned/cfg_args
Config file found: /content/drive/MyDrive/WeatherGS_test/output/tanabata_rain_cleaned/cfg_args
Rendering /content/drive/MyDrive/WeatherGS_test/output/tanabata_rain_cleaned
Loading trained model at iteration 30000 [13/03 05:54:43]
Reading camera 1/211 22222 [13/03 05:54:47]
Camera(id=1, model='PINHOLE', width=1200, height=800, params=array([1539.92539881, 1544.10648914,  600.        ,  400.        ])) [13/03 05:54:47]
PINHOLE [13/03 05:54:47]
Reading camera 2/211 22222 [13/03 05:54:47]
Camera(id=1, model='PINHOLE', width=1200, height=800, params=array([1539.92539881, 1544.10648914,  600.        ,  400.        ])) [13/03 05:54:47]
PINHOLE [13/03 05:54:47]
Reading camera 3/211 22222 [13/03 05:54:48]
Camera(id=1, model='PINHOLE', width=1200, height=800, params=array([1539.92539881, 1544.10648914,  600.        ,  400.        ])) [13/03 05:54:48]
PINHOLE [13/03 05:54:48]
Reading camera 4/21